---
title: Logistic Regression and Regularization
jupyter: python3
---

## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/11-Regression-II-Logistic-Regularization.ipynb)

In [ ]:
#| echo: false
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib as mp
import sklearn
from IPython.display import Image, HTML
import statsmodels.api as sm
from sklearn import model_selection
from sklearn import metrics

import laUtilities as ut

%matplotlib inline

So far we have seen linear regression: 

* a continuous valued observation is estimated as a linear (or affine) function
 of the independent variables.

Now we will look at the following situation.

## Estimating a Probability

::: {.incremental}

* Imagine that you are observing a binary variable -- value 0 or 1.

* That is, these could be pass/fail, admit/reject, Democrat/Republican, etc.

* Assume there is some __probability__ of observing a 1, and that probability is a
function of certain independent variables.

* So the key properties of a problem that make it appropriate for logistic
regression are:

    * You are trying to predict a __categorical__ variable
    * You want to estimate a __probability__ of seeing a particular value of the
      categorical variable.

:::

## Example: Grad School Admission

::: {.content-visible when-profile="web"}

::: {.callout-note}
The following example was adapted from this 
[URL](http://www.ats.ucla.edu/stat/r/dae/logit.htm) which seems to be no longer
available. There is an 
[archive of the page](https://web.archive.org/web/20161118221128/http://www.ats.ucla.edu/stat/r/dae/logit.htm)
and an [archive of the dataset](https://web.archive.org/web/20161022051727/http://www.ats.ucla.edu/stat/data/binary.csv).
:::
:::

Let's consider this question:

> What is the probability I will be admitted to Grad School?

Let's see how variables, such as,

* _GRE_ (Graduate Record Exam scores), 
* _GPA_ (grade point average), and 
* prestige of the undergraduate institution

affect admission into graduate school. 

::: {.content-visible when-profile="slides"}
## Example continued
:::

The response variable, admit/don't admit, is a binary variable.

So there are three predictor variables: __gre,__ __gpa__ and __rank.__ 

* We will treat the variables _gre_ and _gpa_ as continuous. 
* The variable _rank_ takes on the values 1 through 4 with 1 being the highest prestige.

::: {.content-visible when-profile="slides"}
## Example continued
:::

Let's look at 10 lines of the data:

In [ ]:
#| echo: false
# original data source: http://www.ats.ucla.edu/stat/data/binary.csv
df = pd.read_csv('data/ats-admissions.csv') 
df.head(10)

In [ ]:
df.shape

::: {.content-visible when-profile="slides"}
## Example continued
:::

and some summary statistics:

In [ ]:
#| code-fold: true
df.describe()

::: {.content-visible when-profile="slides"}
## Example continued
:::

We can also plot histograms of the variables:

In [ ]:
#| code-fold: true
df.hist(figsize = (10, 4));

> Note how `df.hist()` automatically plots a histogram for each column in the
> dataframe as a subplot.

::: {.content-visible when-profile="slides"}
## Example continued
:::

Let's look at how each independent variable affects admission probability by
plotting the mean admission probability as a function of the independent variable.

> Note that there's a greatly expanded [`groupby`](A3-Pandas.qmd#understanding-pandas-dataframe.groupby) section in the Pandas refresher.

We add error bars to the means to indicate the standard error of the mean.

::: {.content-visible when-profile="slides"}
## Example continued
:::

First, __rank__:

In [ ]:
#| code-fold: true
import numpy as np

# Calculate mean and standard error
grouped = df.groupby('rank')['admit']
means = grouped.mean()

# Compute 'standard error of the mean'
errors = grouped.std() / np.sqrt(grouped.count())

# Plot with error bars
ax = means.plot(marker='o', yerr=errors, fontsize=12, capsize=5)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('Rank', fontsize=16);

::: {.content-visible when-profile="slides"}
## Example continued
:::

Next, __GRE__:

In [ ]:
#| code-fold: true
grouped_gre = df.groupby('gre')['admit']
means_gre = grouped_gre.mean()
errors_gre = grouped_gre.std() / np.sqrt(grouped_gre.count())

ax = means_gre.plot(marker='o', yerr=errors_gre, fontsize=12, capsize=5)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('GRE', fontsize=16);

::: {.content-visible when-profile="slides"}
## Example continued
:::

Finally, __GPA__ (for this visualization, we aggregate GPA into 8 bins):

In [ ]:
#| code-fold: true
bins = np.linspace(df.gpa.min(), df.gpa.max(), 8)
bin_centers = (bins[:-1] + bins[1:]) / 2
grouped_gpa = df.groupby(np.digitize(df.gpa, bins)).mean()['admit']
ax = grouped_gpa.plot(marker='o', fontsize=12)
ax.set_ylabel('P[admit]', fontsize=16)
ax.set_xlabel('GPA', fontsize=16)
ax.set_xticks(range(1, len(bin_centers) + 1))
ax.set_xticklabels([f'{center:.2f}' for center in bin_centers], rotation=45);

::: {.content-visible when-profile="slides"}
## Example continued
:::

Finally, we plot admission status versus GRE score for each data point for each of the four ranks:

In [ ]:
#| code-fold: true
df1 = df[df['rank']==1]
df2 = df[df['rank']==2]
df3 = df[df['rank']==3]
df4 = df[df['rank']==4]

fig = plt.figure(figsize = (10, 5))

ax1 = fig.add_subplot(221)
df1.plot.scatter('gre','admit', ax = ax1)
plt.title('Rank 1 Institutions')

ax2 = fig.add_subplot(222)
df2.plot.scatter('gre','admit', ax = ax2)
plt.title('Rank 2 Institutions')

ax3 = fig.add_subplot(223, sharex = ax1)
df3.plot.scatter('gre','admit', ax = ax3)
plt.title('Rank 3 Institutions')

ax4 = fig.add_subplot(224, sharex = ax2)
plt.title('Rank 4 Institutions')
df4.plot.scatter('gre','admit', ax = ax4);

What we want to do is to fit a model that predicts the probability of admission
as a function of these independent variables.

## Logistic Regression

::: {.incremental}
* Logistic regression is concerned with estimating a __probability.__

* However, all that is available are categorical observations, which we will code as 0/1.

* That is, these could be pass/fail, admit/reject, Democrat/Republican, etc.

* Now, a linear function like $\beta_0 + \beta_1 x$ cannot be used to predict
probability directly, because 
    * the linear function takes on all values (from -$\infty$ to +$\infty$), 
    * and probability only ranges over $[0, 1]$.

:::

## Odds and Log-Odds

However, there is a transformation of probability that works: it is called
__log-odds__.

For any probabilty $p$, the __odds__ is defined as $p/(1-p)$, which is the ratio
of the probability of an event to the probability of the non-event.

Notice that odds vary from 0 to $\infty$, and odds < 1 indicates that $p < 1/2$.

Now, there is a good argument that to fit a linear function, instead of using
odds, we should use log-odds. 

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

That is simply $\log p/(1-p)$ which is also called the __logit__ function, which
is an abbreviation for **log**istic un**it**.

In [ ]:
#| code-fold: true
pvec = np.linspace(0.01, 0.99, 100)
ax = plt.figure(figsize = (6, 4)).add_subplot()
ax.plot(pvec, np.log(pvec / (1-pvec)))
ax.tick_params(labelsize=12)
ax.set_xlabel('Probability', fontsize = 14)
ax.set_ylabel('Log-Odds', fontsize = 14)
ax.set_title('Logit Function: $\log (p/1-p)$', fontsize = 16);

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

So, logistic regression does the following: it does a linear regression of
$\beta_0 + \beta_1 x$ against $\log p/(1-p)$.

That is, it fits:

$$
\begin{aligned}
\beta_0 + \beta_1 x &= \log \frac{p(x)}{1-p(x)} \\
e^{\beta_0 + \beta_1 x} &= \frac{p(x)}{1-p(x)} \quad \text{(exponentiate both sides)} \\
e^{\beta_0 + \beta_1 x} (1-p(x)) &= p(x) \quad \text{(multiply both sides by $1-p(x)$)} \\
e^{\beta_0 + \beta_1 x}  &= p(x) + p(x)e^{\beta_0 + \beta_1 x} \quad \text{(distribute $p(x)$)} \\
\frac{e^{\beta_0 + \beta_1 x}}{1 +e^{\beta_0 + \beta_1 x}} &= p(x)
\end{aligned}
$$

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

So, logistic regression fits a probability of the following form:

$$
p(x) = P(y=1\mid x) = \frac{e^{\beta_0+\beta_1 x}}{1+e^{\beta_0+\beta_1 x}}.
$$

This is a **sigmoid function**; when $\beta_1 > 0$, 

* as $x\rightarrow \infty$, then $p(x)\rightarrow 1$ and 
* as $x\rightarrow -\infty$, then $p(x)\rightarrow 0$.

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

Holding $\beta_0$ constant, we see that as $\beta_1$ increases, the logistic function becomes steeper.

In [ ]:
#| code-fold: true
alphas = [-4, -8,-12,-20]
alphas = [-8, -8, -8, -8]
betas = [0.2,0.4,0.6,1]
x = np.arange(40)
fig = plt.figure(figsize=(8, 6)) 
ax = plt.subplot(111)

for i in range(len(alphas)):
    a = alphas[i]
    b = betas[i]
    y = np.exp(a+b*x)/(1+np.exp(a+b*x))
#     plt.plot(x,y,label=r"$\frac{e^{%d + %3.1fx}}{1+e^{%d + %3.1fx}}\;\beta_0=%d, \beta_1=%3.1f$" % (a,b,a,b,a,b))
    ax.plot(x,y,label=r"$\beta_0=%d,$    $\beta_1=%3.1f$" % (a,b))
ax.tick_params(labelsize=12)
ax.set_xlabel('x', fontsize = 14)
ax.set_ylabel('$p(x)$', fontsize = 14)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop={'size': 16})
ax.set_title('Logistic Functions', fontsize = 16);

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

Holding $\beta_1$ constant, we see that as $\beta_0$ increases, the logistic function shifts to the right.

In [ ]:
#| code-fold: true
alphas = [-4, -8,-12,-20]
betas = [0.4, 0.4, 0.4, 0.4]
x = np.arange(40)
fig = plt.figure(figsize=(8, 6)) 
ax = plt.subplot(111)

for i in range(len(alphas)):
    a = alphas[i]
    b = betas[i]
    y = np.exp(a+b*x)/(1+np.exp(a+b*x))
#     plt.plot(x,y,label=r"$\frac{e^{%d + %3.1fx}}{1+e^{%d + %3.1fx}}\;\beta_0=%d, \beta_1=%3.1f$" % (a,b,a,b,a,b))
    ax.plot(x,y,label=r"$\beta_0=%d,$    $\beta_1=%3.1f$" % (a,b))
ax.tick_params(labelsize=12)
ax.set_xlabel('x', fontsize = 14)
ax.set_ylabel('$p(x)$', fontsize = 14)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop={'size': 16})
ax.set_title('Logistic Functions', fontsize = 16);

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

Varying both $\beta_0$ and $\beta_1$ gives us a more general logistic function.


In [ ]:
#| code-fold: true
alphas = [-4, -8,-12,-20]
betas = [0.2,0.4,0.6,1]
x = np.arange(40)
fig = plt.figure(figsize=(8, 6)) 
ax = plt.subplot(111)

for i in range(len(alphas)):
    a = alphas[i]
    b = betas[i]
    y = np.exp(a+b*x)/(1+np.exp(a+b*x))
#     plt.plot(x,y,label=r"$\frac{e^{%d + %3.1fx}}{1+e^{%d + %3.1fx}}\;\beta_0=%d, \beta_1=%3.1f$" % (a,b,a,b,a,b))
    ax.plot(x,y,label=r"$\beta_0=%d,$    $\beta_1=%3.1f$" % (a,b))
ax.tick_params(labelsize=12)
ax.set_xlabel('x', fontsize = 14)
ax.set_ylabel('$p(x)$', fontsize = 14)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop={'size': 16})
ax.set_title('Logistic Functions', fontsize = 16);

Parameter $\beta_1$ controls how fast $p(x)$ raises from $0$ to $1$

The value of -$\beta_0$/$\beta_1$ shows the value of $x$ for which $p(x)=0.5$

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

Another interpretation of $\beta_0$ is that it gives the __base rate__ -- the
unconditional probability of a 1.   

That is, if you knew nothing about a
particular data item, then $p(x) = 1/(1+e^{-\beta_0})$.

In [ ]:
#| fig-align: center

# plot the base rate as a function of beta_0
beta_0_values = np.linspace(-10, 10, 100)
base_rate = 1 / (1 + np.exp(-beta_0_values))
plt.figure(figsize=(6, 4))
plt.plot(beta_0_values, base_rate)
plt.xlabel('beta_0')
plt.ylabel('Base Rate')
plt.title('Base Rate as a function of beta_0')
plt.show()

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

The function $f(x) = \log (x/(1-x))$ is called the __logit__ function.

So a compact way to describe logistic regression is that it finds regression
coefficients $\beta_0, \beta_1$ to fit:

$$
\text{logit}\left(p(x)\right)=\log\left(\frac{p(x)}{1-p(x)} \right) = \beta_0 + \beta_1 x.
$$

Note also that the __inverse__ logit function is:

$$
\text{logit}^{-1}(x) = \frac{e^x}{1 + e^x}.
$$

Somewhat confusingly, this is called the __logistic__ function.

::: {.content-visible when-profile="slides"}
## Odds and Log-Odds
:::

So, the best way to think of logistic regression is that we compute a linear function:
    
$$
\beta_0 + \beta_1 x,
$$
    
and then *map* that to a probability using the inverse $\text{logit}$ function:

$$
\frac{e^{\beta_0+\beta_1 x}}{1+e^{\beta_0+\beta_1 x}}.
$$

## Logistic vs Linear Regression

Let's take a moment to compare linear and logistic regression.

In __Linear regression__ we fit 

$$
y_i = \beta_0 +\beta_1 x_i + \epsilon_i.
$$

We do the fitting by minimizing the sum of squared errors $\Vert\epsilon\Vert$.
This can be done in closed form using either geometric arguments or by calculus.

Now, if $\epsilon_i$ comes from a normal distribution with mean zero and some
fixed variance, then minimizing the sum of squared errors is exactly the same as finding the
maximum likelihood of the data with respect to the probability of the errors.

So, in the case of linear regression, it is a lucky fact that the __MLE__ of
$\beta_0$ and $\beta_1$ can be found by a __closed-form__ calculation.

::: {.content-visible when-profile="slides"}
## Logistic vs Linear Regression
:::

In __Logistic regression__ we fit 

$$
\text{logit}(p(x_i)) = \beta_0 + \beta_1 x_i.
$$


with $\text{P}(y_i=1\mid x_i)=p(x_i).$

How should we choose parameters?   

Here too, we use Maximum Likelihood Estimation of the parameters.

That is, we choose the parameter values that maximize the likelihood of the data given the model.

$$
\text{P}(y_i \mid x_i) = 
\left\{\begin{array}{lr}\text{logit}^{-1}(\beta_0 + \beta_1 x_i)& \text{if } y_i = 1\\
1 - \text{logit}^{-1}(\beta_0 + \beta_1 x_i)& \text{if } y_i = 0\end{array}\right.
$$

::: {.content-visible when-profile="slides"}
## Logistic vs Linear Regression
:::

We can write this as a single expression:

$$
\text{P}(y_i \mid x_i) = \text{logit}^{-1}(\beta_0 + \beta_1 x_i)^{y_i} (1-\text{logit}^{-1}(\beta_0 + \beta_1 x_i))^{1-y_i},
$$

where we assume the parameters are fixed.

We can reinterpret this to express the __likelihood__ of parameters $\beta_0$, $\beta_1$:

$$
L(\beta_0, \beta_1 \mid x_i, y_i) = \text{logit}^{-1}(\beta_0 + \beta_1 x_i)^{y_i} (1-\text{logit}^{-1}(\beta_0 + \beta_1 x_i))^{1-y_i},
$$

given that we have observed the data $(x_i, y_i)$.

**This is our objective function to maximize.**

However, there is no closed-form solution so we optimize it numerically with gradient descent.


## How Gradient Descent Works

**Algorithm:**

1. **Initialize** parameters: Start with random values $\beta^{(0)}$

2. **Compute gradient**: Calculate $\nabla \ell(\beta^{(t)})$ - the direction of steepest increase

3. **Update parameters**: Take a step in that direction:
   $$\beta^{(t+1)} = \beta^{(t)} + \alpha \nabla \ell(\beta^{(t)})$$
   where $\alpha$ is the **learning rate** (step size)

4. **Repeat** steps 2-3 until convergence (gradient $\approx 0$ or max iterations reached)

**Result:** Parameters that (locally) maximize the likelihood of the observed data.

## Logistic Regression In Practice

So, in summary, we have:

**Input** pairs $(x_i,y_i)$

**Output** parameters $\widehat{\beta_0}$ and $\widehat{\beta_1}$ that maximize the
likelihood of the data given these parameters for the logistic regression model.

**Method** Maximum likelihood estimation, obtained by gradient descent.

The standard package will give us a coefficient $\beta_i$ for each
independent variable (feature).

::: {.content-visible when-profile="slides"}
## Logistic Regression in Practice
:::

If we want to include a constant (i.e., $\beta_0$) we need to add a column of 1s (just
like in linear regression).

In [ ]:
#| code-fold: true
df['intercept'] = 1.0
train_cols = df.columns[1:]
train_cols

In [ ]:
#| code-fold: true
logit = sm.Logit(df['admit'], df[train_cols])
 
# fit the model
result = logit.fit() 

::: {.content-visible when-profile="slides"}
## Logistic Regression in Practice
:::

Statsmodels gives us a summary of the model fit.

In [ ]:
#| code-fold: true
result.summary()

Notice that all of our independent variables are considered significant (no
confidence intervals contain zero).

## Using the Model

Note that by fitting a model to the data, we can make predictions for inputs that
were not in the training data.  

Furthermore, we can make a prediction of a probability for cases where we don't
have enough data to estimate the probability directly -- e.g., for specific
parameter values.

Let's see how well the model fits the data.

::: {.content-visible when-profile="slides"}
## Using the Model
:::

We have three independent variables, so in each case we'll use average values
for the two that we aren't evaluating.

GPA (GRE = 600, Rank = 2.5):

In [ ]:
#| code-fold: true
#| fig-align: center
bins = np.linspace(df.gpa.min(), df.gpa.max(), 10)
groups = df.groupby(np.digitize(df.gpa, bins))
prob = [result.predict([600, b, 2.5, 1.0]) for b in bins]
ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(bins, prob)
ax.plot(bins,groups.admit.mean(),'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('gpa', fontsize = 14)
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of GPA', fontsize = 16);

::: {.content-visible when-profile="slides"}
## Logistic Regression in Practice
:::

GRE Score (GPA = 3.4, Rank = 2.5):

In [ ]:
#| code-fold: true
#| fig-align: center
prob = [result.predict([b, 3.4, 2.5, 1.0]) for b in sorted(df.gre.unique())]
ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(sorted(df.gre.unique()), prob)
ax.plot(df.groupby('gre').mean()['admit'],'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('gre', fontsize = 14)
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of GRE', fontsize = 16);

::: {.content-visible when-profile="slides"}
## Logistic Regression in Practice
:::

Institution Rank (GRE = 600, GPA = 3.4):

In [ ]:
#| code-fold: true
#| fig-align: center
prob = [result.predict([600, 3.4, b, 1.0]) for b in range(1,5)]
ax = plt.figure(figsize = (7, 4)).add_subplot()
ax.plot(range(1,5), prob)
ax.plot(df.groupby('rank').mean()['admit'],'o')
ax.tick_params(labelsize=12)
ax.set_xlabel('Rank', fontsize = 14)
ax.set_xlim([0.5,4.5])
ax.set_ylabel('P[admit]', fontsize = 14)
ax.set_title('Marginal Effect of Rank', fontsize = 16);

## Logistic Regression in Perspective

At the start of lecture we emphasized that logistic regression is concerned with
estimating a __probability__ model for __discrete__ (0/1) data. 

However, it may well be the case that we want to do something with the
probability that amounts to __classification.__

For example, we may classify data items using a rule such as "Assign item $x_i$
to Class 1 if $p(x_i) > 0.5$".

For this reason, logistic regression could be considered a classification method.

::: {.content-visible when-profile="slides"}
## Logistic Regression in Perspective
:::

Let's use our logistic regression as a classifier.

We want to ask whether we can correctly predict whether a student gets admitted
to graduate school.

Let's separate our training and test data:

In [ ]:
#| code-fold: true
X_train, X_test, y_train, y_test = model_selection.train_test_split(
        df[train_cols], df['admit'],
        test_size=0.4, random_state=1)

::: {.content-visible when-profile="slides"}
## Logistic Regression in Perspective
:::

Now, there are some standard metrics used when evaluating a binary classifier.

Let's say our classifier is outputting "yes" when it thinks the student will be admitted.

There are four cases:

* Classifier says "yes", and student __is__ admitted:  __True Positive.__
* Classifier says "yes", and student __is not__ admitted:  __False Positive.__
* Classifier says "no", and student __is__ admitted:  __False Negative.__
* Classifier says "no", and student __is not__ admitted:  __True Negative.__

::: {.content-visible when-profile="slides"}
## Logistic Regression in Perspective
:::

__Precision__ is the fraction of "yes" classifications that are correct:

$$
\mbox{Precision} = \frac{\mbox{True Positives}}{\mbox{True Positives + False Positives}}.
$$
    
__Recall__ is the fraction of admits that we say "yes" to:

$$
\mbox{Recall} = \frac{\mbox{True Positives}}{\mbox{True Positives + False Negatives}}.
$$

In [ ]:
#| code-fold: true
def evaluate(y_train, X_train, y_test, X_test, threshold):

    # learn model on training data
    logit = sm.Logit(y_train, X_train)
    result = logit.fit(disp=False)
    
    # make probability predictions on test data
    y_pred = result.predict(X_test)
    
    # threshold probabilities to create classifications
    y_pred = y_pred > threshold
    
    # report metrics
    precision = metrics.precision_score(y_test, y_pred)
    recall = metrics.recall_score(y_test, y_pred)
    return precision, recall

precision, recall = evaluate(y_train, X_train, y_test, X_test, 0.5)

print(f'Precision: {precision:0.3f}, Recall: {recall:0.3f}')

::: {.content-visible when-profile="slides"}
## Logistic Regression in Perspective
:::

Now, let's get a sense of average accuracy:

In [ ]:
#| code-fold: true
PR = []
for i in range(20):
    X_train, X_test, y_train, y_test = model_selection.train_test_split(
            df[train_cols], df['admit'],
            test_size=0.4)
    PR.append(evaluate(y_train, X_train, y_test, X_test, 0.5))

In [ ]:
#| code-fold: true
avgPrec = np.mean([f[0] for f in PR])
avgRec = np.mean([f[1] for f in PR])
print(f'Average Precision: {avgPrec:0.3f}, Average Recall: {avgRec:0.3f}')

::: {.content-visible when-profile="slides"}
## Logistic Regression in Perspective
:::

Sometimes we would like a single value that describes the overall performance of
the classifier.

For this, we take the harmonic mean of precision and recall, called __F1 Score__:

$$
\mbox{F1 Score} = 2 \;\;\frac{\mbox{Precision} \cdot \mbox{Recall}}{\mbox{Precision} + \mbox{Recall}}.
$$

::: {.content-visible when-profile="slides"}
## F1 Score as a function of threshold
:::

Using this, we can evaluate other settings for the threshold.

In [ ]:
#| code-fold: true
import warnings
warnings.filterwarnings("ignore")
def evalThresh(df, thresh):
    PR = []
    for i in range(20):
        X_train, X_test, y_train, y_test = model_selection.train_test_split(
                df[train_cols], df['admit'],
                test_size=0.4)
        PR.append(evaluate(y_train, X_train, y_test, X_test, thresh))
    avgPrec = np.mean([f[0] for f in PR])
    avgRec = np.mean([f[1] for f in PR])
    return 2 * (avgPrec * avgRec) / (avgPrec + avgRec), avgPrec, avgRec

tvals = np.linspace(0.05, 0.8, 50)
f1vals = [evalThresh(df, tval)[0] for tval in tvals]

In [ ]:
#| code-fold: true
#| fig-align: center
plt.figure(figsize=(6, 3))
plt.plot(tvals,f1vals)
plt.ylabel('F1 Score')
plt.xlabel('Threshold for Classification')
plt.title('F1 as a function of Threshold');

Based on this plot, we can say that the best classification threshold appears to
be around 0.3, where precision and recall are:

In [ ]:
#| code-fold: true
F1, Prec, Rec = evalThresh(df, 0.3)
print('Best Precision: {:0.3f}, Best Recall: {:0.3f}'.format(Prec, Rec))

::: {.content-visible when-profile="web"}

The example here is based on
http://blog.yhathq.com/posts/logistic-regression-and-python.html
where you can find additional details.

:::

## ROC-AUC Curve

::: {.columns}
::: {.column width="50%"}

<br><br>

* The ROC-AUC curve is a plot of the true positive rate against the false positive rate at various threshold settings.

* The AUC is the area under the ROC curve.

* The AUC is a measure of the overall performance of the classifier.

:::
::: {.column width="50%"}

In [ ]:
#| code-fold: true
#| fig-align: center

# Get predicted probabilities from the model
y_pred_proba = result.predict(df[train_cols])

# Compute ROC curve and AUC
fpr, tpr, thresholds = metrics.roc_curve(df['admit'], y_pred_proba)
auc_score = metrics.roc_auc_score(df['admit'], y_pred_proba)

# Plot ROC curve
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve for Logistic Regression Model', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_score:.4f}")

:::
:::

## Recap

* Logistic regression is used to predict a probability.
* It is a linear model for the log-odds.
* It is fit by maximum likelihood.
* It can be evaluated as a classifier.

## From logistic regression to regularization

::: {.incremental}
* We can now fit **linear** models (last lecture) and **logistic** models (just now), both by choosing coefficients $\boldsymbol{\beta}$ that minimize a loss.
* Both can **overfit**, exactly as [Generalization](07-Generalization.qmd) warned: with many (or correlated) features, the fitted $\boldsymbol{\beta}$ chases noise in the training set and generalizes poorly.
* **Regularization** is the fix -- add a penalty on the size of $\boldsymbol{\beta}$ to the loss -- and it is the *same idea* for linear regression, logistic regression, and later neural networks.
* Part 2 develops it for linear regression, where the geometry is easiest to see.
:::

# Part 2: Regularization

In [ ]:
#| echo: false
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib as mp
import sklearn
import statsmodels.api as sm
from sklearn import model_selection
from sklearn import metrics

import laUtilities as ut

from statsmodels.sandbox.regression.predstd import wls_prediction_std
from statsmodels.regression.linear_model import OLS
import statsmodels.formula.api as smf

import warnings

np.random.seed(9876789)

In [ ]:
from statsmodels.datasets.longley import load_pandas
y = load_pandas().endog
X = load_pandas().exog
X['const'] = 1.0
X.index = X['YEAR']
y.index = X['YEAR']
X.drop('YEAR', axis = 1, inplace = True)
print("X.head()")
print(X.head())
print("\n\ny.head()")
print(y.head())

---

An important warning is issued stating the condition number is large. What does this mean?

In [ ]:
ols_model = sm.OLS(y, X)
ols_results = ols_model.fit()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(ols_results.summary())

## Condition Number

The notion of conditioning pertains to the perturbation behavior of a mathematical problem. A well-conditioned problem is one where small changes to the inputs produce small changes to the output. An ill-conditioned problem is one where small changes in the input can produce very large changes in the output. 

The condition number of a matrix provides an indication of how accurately you can compute with it. 

A large condition number tells us that our problem is ill conditioned, i.e., small changes to the input can produce very large changes in the output.

A small condition number tells us that our problem is well-conditioned.

---

The condition number is derived using matrix norms, which is beyond the scope of this course. However, we will use the following definition for the condition number of our matrix

$$
\kappa(X) = \frac{\sigma_{\text{max}}}{\sigma_{\text{min}}},
$$

where $\sigma_{\text{max}}$, $\sigma_{\text{min}}$ are the maximum and minimum singular values, respectively, of the design matrix $X$.

The SVD again provides us with important properties of a matrix.

---

Another important fact is that

$$
\kappa(X^TX) = \frac{\sigma_{\text{max}}^2}{\sigma_{\text{min}}^2}.
$$

This means that if we have a poorly conditioned data matrix $X$, then $X^TX$ is even more poorly conditioned.

This is why you should never work directly with $X^TX$ as it can be numerically unstable.

## Normal Equations

To solve the least-squares problem we solve the normal equations

$$
X^TX\boldsymbol{\beta} = X^Ty.
$$

These equations always have at least one solution. However, the *at least one* part is problematic.

If there are multiple solutions, they are in a sense all equivalent in that they yield the same value of $\Vert X\boldsymbol{\beta} - y\Vert_2$.

However, the actual values of $\boldsymbol{\beta}$ can vary tremendously and so it is not clear how best to interpret which solution is actually the best.

When does this problem occur? 

## Linear Dependence

It occurs when $X^TX$ is __not invertible.__

This happens when the columns of $X$ are linearly dependent -- that is, one column can be expressed as a linear combination of the other columns.

In that case, it is not possible to solve the normal equations by computing $\hat{\boldsymbol{\beta}} \neq (X^TX)^{-1}X^Ty.$

This is the simplest kind of __multicollinearity__.

---

What are the implications of a matrix not being invertible on the condition number?

If a matrix $Z = X^{T}X$ is not invertible, there is a zero singular value. This implies

$$
\kappa(Z) = \infty.
$$

In other words, the problem of solving an equation with a non-invertible matrix is completely ill-conditioned. In fact, it's a problem that is impossible to solve.

## Near Linear Dependence

Near linear dependence causes problems as well. This can happen, for example, due to  measurement errors. Or when two or more columns are __strongly correlated__.

In such a situation, we have some column of our design matrix that is __close to__ being a linear combination of the other columns.

When these situations occur we will have problems with linear regression.

---

As a result of near linear dependence, the smallest singular value of the design matrix $X$ will be close to zero. This means that $\kappa(X)$ will be very large.

The condition number tells us that a small change in the input to our problem can result in large changes to the output. 

This means that for a design matrix $X$ with near linearly dependent columns, the values we compute for $\boldsymbol{\beta}$ in our linear regression can vary significantly.

This is why we see the addition of a regularization (penalty) term involving $\boldsymbol{\beta}$ in the least squares minimization problem. This process *regularizes* the solution $\boldsymbol{\beta}$.


## Longley Dataset

Recall that the condition number of our data is around $10^8$. 

A large condition number is evidence of a problem.

As a general rule of thumb:

* If the condition number is less than 100, there is no serious problem
with multicollinearity.
* Condition numbers between 100 and 1000 imply moderate to strong multicollinearity.
* Condition numbers bigger than 1000 indicate severe multicollinearity.

---

Let's look at pairwise scatter plots of the Longley data.

In [ ]:
sns.pairplot(X[['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP']])
plt.show()

We can see __very__ strong linear relationships between, e.g., __GNP Deflator__, __GNP__, and __Population.__

## Addressing Multicollinearity

Here are two strategies we can employ to address multicollinearity:

1. Ridge Regression
2. Model Selection via LASSO

::: {.aside}
PCA also addresses multicollinearity by transforming the correlated features into uncorrelated features. However in this approach you lose the original features, which is less explainable.
:::

## Ridge Regression

The first thing to note is that when columns of $X$ are nearly dependent, the components of $\hat{\boldsymbol{\beta}}$ tend to be __large in magnitude__.

:::: {.columns}
::: {.column width="50%"}

In [ ]:
#| fig-align: center
ax = ut.plotSetup(size=(4,2))
ut.centerAxes(ax)
u = np.array([1, 2])
v = np.array([4, 1])
alph = 1.6
beta = -1.25
sum_uv = (alph * u) + (beta * v)
ax.arrow(0, 0, u[0], u[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, v[0], v[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.text(sum_uv[0]-.5, sum_uv[1]+0.25, r'$\mathbf{y}$',size=12)
ax.text(u[0]+0.25, u[1]-0.25, r'${\bf u}$', size=12)
ax.text(v[0]+0.25, v[1]+0.25, r'${\bf v}$',size=12)
ut.plotPoint(ax, sum_uv[0], sum_uv[1])
ax.plot(0, 0, '')
plt.show()

Consider a regression in which we are predicting the point $\mathbf{y}$ as a linear function of two $X$ columns, which we'll denote $\mathbf{u}$ and $\mathbf{v}$.
:::
::: {.column width="50%"}

In [ ]:
#| fig-align: center
ax = ut.plotSetup(size=(4, 2))
ut.centerAxes(ax)
u = np.array([1, 2])
v = np.array([4, 1])
alph = 1.6
beta = -1.25
sum_uv = (alph * u) + (beta * v)
ax.arrow(0, 0, u[0], u[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, v[0], v[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, alph * u[0], alph * u[1], head_width=0.2, 
         head_length=0.2, length_includes_head = True)
ax.arrow(alph * u[0], alph * u[1], sum_uv[0] - alph * u[0], sum_uv[1] - alph * u[1], 
         head_width=0.2, 
         head_length=0.2, length_includes_head = True, color = 'r')
ax.text(sum_uv[0]-2, sum_uv[1]+0.25, r'$\beta_1{\bf u}$+$\beta_2{\bf v}$',size=12)
ax.text(u[0]+0.25, u[1]-0.25, r'${\bf u}$', size=12)
ax.text(alph * u[0]+0.25, alph * u[1]-0.25, r'$\beta_1{\bf u}$', size=12)
ax.text(-2, 2.75, r'$\beta_2{\bf v}$', size=12)
ax.text(v[0]+0.25, v[1]+0.25, r'${\bf v}$',size=12)
ut.plotPoint(ax, sum_uv[0], sum_uv[1])
ax.plot(0, 0, '')
plt.show()

We determine the coefficients $\beta_1$ and $\beta_2$.
:::
::::

---

Now consider if the columns of $X$ are __nearly dependent__.

:::: {.columns}
::: {.column width="50%"}

In [ ]:
#| fig-align: center
ax = ut.plotSetup(size=(4, 2))
ut.centerAxes(ax)
u = np.array([2, 1])
v = np.array([4, 1])
ax.arrow(0, 0, u[0], u[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, v[0], v[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.text(sum_uv[0]-.5, sum_uv[1]+0.25, r'$\mathbf{y}$',size=12)
ax.text(u[0]+0.25, u[1]-0.25, r'${\bf u}$', size=12)
ax.text(v[0]+0.25, v[1]+0.25, r'${\bf v}$',size=12)
ut.plotPoint(ax, sum_uv[0], sum_uv[1])
ax.plot(0, 0, '')
plt.show()

:::
::: {.column width="50%"}

In [ ]:
#| fig-align: center
ax = ut.plotSetup(size=(4, 2))
ut.centerAxes(ax)
u = np.array([2, 1])
v = np.array([4, 1])
alph = 2.675
beta = -8.75
ax.arrow(0, 0, u[0], u[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, v[0], v[1], head_width=0.2, head_length=0.2, length_includes_head = True)
ax.arrow(0, 0, alph * u[0], alph * u[1], head_width=0.2, 
         head_length=0.2, length_includes_head = True)
ax.arrow(alph * u[0], alph * u[1], sum_uv[0] - alph * u[0], sum_uv[1] - alph * u[1], 
         head_width=0.2, 
         head_length=0.2, length_includes_head = True, color = 'r')
ax.text(sum_uv[0]-2, sum_uv[1]+0.25, r'$\beta_1{\bf u}$+$\beta_2{\bf v}$',size=12)
ax.text(u[0]+0.25, u[1]-0.25, r'${\bf u}$', size=12)
ax.text(alph * u[0]+0.25, alph * u[1]-0.25, r'$\beta_1{\bf u}$', size=12)
ax.text(-2, 2.75, r'$\beta_2{\bf v}$', size=12)
ax.text(v[0]+0.25, v[1]+0.25, r'${\bf v}$',size=12)
ut.plotPoint(ax, sum_uv[0], sum_uv[1])
ax.plot(0, 0, '')
plt.show()

:::
::::

If you imagine the values of $\beta_1$ and $\beta_2$ necessary to create $\mathbf{y} = \beta_1{\bf u}$+$\beta_2{\bf v}$, you can see that $\beta_1$ and $\beta_2$ will be __very large__ in magnitude.

This geometric argument illustrates why the regression coefficients will be very large under multicollinearity.

As a result, the value of $\Vert\boldsymbol{\beta}\Vert_2$ will be very large.

## Ridge Regression

Ridge regression adjusts the least squares regression by shrinking the estimated coefficients towards zero.

The purpose is to fix the magnitude inflation of $\Vert\boldsymbol{\beta}\Vert_2$.

To do this, Ridge regression assumes that the model has no intercept term -- both the response and the predictors have been centered so that $\beta_0 = 0$.

Ridge regression then consists of adding a penalty term to the regression:

$$ 
\hat{\boldsymbol{\beta}} = \arg \min_\boldsymbol{\beta} \Vert X\boldsymbol{\beta} - y \Vert_2^2 + c\Vert\boldsymbol{\beta}\Vert_2^2.
$$

---

For any given $c$ this has a closed-form solution in which $\hat{\boldsymbol{\beta}} = (X^TX +cI)^{−1}X^T\mathbf{y}.$

The solution to the Ridge regression problem always exists and is unique, even when the data contains multicollinearity.

Here, $c \geq 0$ is a tradeoff parameter and controls the strength of the penalty term:

* When $c = 0$, we get the least squares estimator: $\hat{\boldsymbol{\beta}} = (X^TX)^{−1}X^T\mathbf{y}$
* When $c \rightarrow \infty$, we get $\hat{\boldsymbol{\beta}} \rightarrow 0.$
* Increasing the value of $c$ forces the norm of $\hat{\boldsymbol{\beta}}$ to decrease, yielding smaller coefficient estimates in magnitude.

For a finite, positive value of $c$, we are balancing two tasks: fitting
a linear model and shrinking the coefficients.

The coefficient $c$ is a __hyperparameter__ that controls the model complexity. We typically set $c$ by holding out data, i.e., __cross-validation.__

## Scaling

Note that the penalty term $\Vert\boldsymbol{\beta}\Vert_2^2$ would be unfair to the different predictors if they are not on the same scale. 

Therefore, if we know that the variables are not measured in the same units, we typically first perform unit normal scaling on the columns of $X$ and on $\mathbf{y}$ (to standardize the predictors), and then perform ridge regression.

Note that by scaling $\mathbf{y}$ to have zero-mean, we do not need (or include) an intercept in the model.

Another name for ridge regression is __Tikhanov regularization__. You may see this terminology used in textbooks on optimization.

::: {.aside}
Normalizing is needed for this specific method. This is in contrast to the [linear regression](10-Regression-I-Linear.qmd) lecture where we allowed the coefficients to correct for the scaling differences between different units of measure.
:::

---

Here is the performance of Ridge regression on the Longley data.

We are training on half of the data and using the other half for testing.

In [ ]:
#| fig-align: center
from sklearn.metrics import r2_score
nreps = 1000

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_std = scaler.fit_transform(X[['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP']])
y_std = scaler.fit_transform(y.values.reshape(-1, 1))

np.random.seed(1)

vals = []
for alpha in np.r_[np.array([0]), 10**np.linspace(-8.5, -0.5, 20)]:
    res = []
    for rep in range(nreps):
        X_train, X_test, y_train, y_test = model_selection.train_test_split(
            X_std, y_std,
            test_size=0.5)
        model = sm.OLS(y_train, X_train)
        results = model.fit_regularized(alpha = alpha, L1_wt = 0)
        y_oos_predict = results.predict(X_test)
        r2_test = r2_score(y_test, y_oos_predict)
        res.append(r2_test)
    vals.append([alpha, np.mean(res), np.std(res)/np.sqrt(nreps)])

results = np.array(vals)

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (6, 4)).add_subplot()
ax.errorbar(np.log10(results[1:][:, 0]), results[1:][:, 1], 
            results[1:][:, 2],
            label = 'Ridge Regression')
ax.hlines(results[0,1], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dashed',
          label = 'Without Regularization')
ax.hlines(results[0,1]+results[0,2], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dotted')
ax.hlines(results[0,1]-results[0,2], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dotted')
ax.tick_params(labelsize=12)
ax.set_ylabel('$R^2$', fontsize = 14)
plt.legend(loc = 'best')
ax.set_xlabel('$\\log_{10}(c)$', fontsize = 14)
ax.set_title('Ridge Regression Accuracy on Longley Data', fontsize = 16)
plt.show()

--- 

To sum up the idea behind Ridge regression: 

1. There may be many $\boldsymbol{\beta}$ values that are consistent with the equations.   
1. Over-fit $\boldsymbol{\beta}$ values tend to have large magnitudes.
1. We add the regularization term $c \Vert \boldsymbol{\beta}\Vert_2^2$ to the least squares to avoid those solutions.
1. We tune $c$ to an appropriate value via cross-validation.

## Model Selection

Of course, one might attack the problem of multicollinearity as follows:
    
- Multicollinearity occurs when variables (features) are close to linearly dependent.
- These variables do not contribute anything *meaningful* to the quality of the model
- As a result why not simply remove variables from the model that are nearly linearly dependent?

We create a new model when we remove these variables from our regression.

This strategy is called **model selection**.

---

One of the advantages of model selection is __interpretability__: by eliminating variables, we get a clearer picture of the relationship between truly useful features and dependent variables.

However, there is a big challenge inherent in model selection. In general, the possibilities to consider are exponential in the number of features.

That is, if we have $n$ features to consider, then there are $2^n-1$ possible models that incorporate one or more of those features. This space is usually too big to search directly.

Can we use Ridge regression for this problem?

:::: {.fragment}
Ridge regression does not set any coefficients exactly to zero unless $c\rightarrow \infty$, in which case they’re all zero. 

This means Ridge regression cannot perform variable selection. Even though it performs well in terms of prediction accuracy, it does not offer a clear interpretation.
::::


## The LASSO

LASSO differs from Ridge regression __only in terms of the norm__ used by the penalty term.

$$ 
\hat{\beta} = \arg \min_\beta \Vert X\beta - y \Vert_2^2 + c \Vert\beta\Vert_1.
$$

However, this small change in the norm makes a __big difference__ in practice.

The nature of the $\ell_1$ penalty will cause some coefficients to be shrunken to zero exactly.

This means that LASSO can perform model selection by telling us which variables to keep and which to set aside.

As $c$ increases, more coefficients are set to zero, i.e., fewer variables are selected.

In terms of prediction error, LASSO performs comparably to Ridge regression but it has a __big advantage with respect to interpretation.__

---

In [ ]:
#| fig-align: center
from sklearn.metrics import r2_score
nreps = 200

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_std = scaler.fit_transform(X[['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP']])
X_std = np.column_stack([X_std, np.ones(X_std.shape[0])])
y_std = scaler.fit_transform(y.values.reshape(-1, 1))

np.random.seed(1)

vals = []
mean_params = []
for alpha in np.r_[np.array([0]), 10**np.linspace(-5, -0.75, 10)]:
    res = []
    params = []
    for rep in range(nreps):
        X_train, X_test, y_train, y_test = model_selection.train_test_split(
            X_std, y_std,
            test_size=0.5)
        model = sm.OLS(y_train, X_train)
        results = model.fit_regularized(alpha = alpha, L1_wt = 1.0)
        y_oos_predict = results.predict(X_test)
        r2_test = r2_score(y_test, y_oos_predict)
        res.append(r2_test)
        params.append(results.params)
    vals.append([alpha, np.mean(res), np.std(res)/np.sqrt(nreps)])
    mean_params.append(np.r_[alpha, np.mean(params, axis = 0)])
results = np.array(vals)
mean_params = np.array(mean_params)

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (6, 4)).add_subplot()
ax.errorbar(np.log10(results[1:][:, 0]), results[1:][:, 1], 
            results[1:][:, 2],
            label = 'LASSO Regression')
ax.hlines(results[0,1], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dashed',
          label = 'Without Regularization')
ax.hlines(results[0,1]+results[0,2], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dotted')
ax.hlines(results[0,1]-results[0,2], np.log10(results[1, 0]), 
           np.log10(results[-1, 0]), linestyles = 'dotted')
ax.tick_params(labelsize=12)
ax.set_ylabel('$R^2$', fontsize = 14)
#ax.set_xlim([-4, -1])
plt.legend(loc = 'best')
ax.set_xlabel('$\\log_{10}(c)$', fontsize = 14)
ax.set_title('LASSO Accuracy on Longley Data', fontsize = 16)
plt.show()

---

In [ ]:
df = pd.DataFrame(mean_params, columns = ['$\log_{10}(c)$', 'GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP', 'const'])
param_df = df[['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP', 'const']].iloc[1:].copy()
param_df.index = np.log10(df.iloc[1:]['$\log_{10}(c)$'])

In [ ]:
#| fig-align: center
param_df.plot()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop={'size': 16})
plt.title('LASSO Coefficients vs $c$')
plt.show()

--- 

We can use the statsmodel `smf` sub-module to directly type formulas and expressions in the functions of the models. This allows us to, among other things,

- specify the name of the columns to be used to predict another column
- remove columns
- infer the type of the variable (e.g., categorical, numerical)
- apply functions to columns

The `smf` submodule makes use of the [patsy](https://patsy.readthedocs.io/en/latest/) package. patsy is a Python package for describing statistical models (especially linear models, or models that have a linear component) and building design matrices. It is closely inspired by and compatible with the formula mini-language used in R and S.

In the following code cells we will see the syntax that is used to specify columns in the models and how to remove columns from our model

---

Here is an example where we specify the name of the columns to be used to predict another column.

In [ ]:
X['TOTEMP'] = y

In [ ]:
mod = smf.ols(formula='TOTEMP ~ GNPDEFL + GNP + UNEMP + ARMED + POP', data=X)
res = mod.fit()   
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(res.summary())

---

The formula

`formula='TOTEMP ~ GNPDEFL + GNP + UNEMP + ARMED + POP'`

is an R-style formula string that specifies the model.

The variable `TOTEMP` is the dependent variable, which is Total Employment in the Longley dataset.

The syntax `~` separates the dependent variable from the independent variables.

The sytnax `GNPDEFL + GNP + UNEMP + ARMED + POP` are the independent variables, which are GNP Deflator, Gross National Product, Number of Unemployed, Size of the Armed Forces, and Population.

---

This is an example where we remove columns from the data and exclude the y-intercept.

In [ ]:
mod = smf.ols(formula='TOTEMP ~ GNPDEFL + GNP + UNEMP - 1', data=X)
res = mod.fit()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(res.summary())

---

The formula is 

`formula='TOTEMP ~ GNPDEFL + GNP + UNEMP - 1'`

We still have the same dependent variable `TOTEMP`. The independent variables are
`GNPDEFL + GNP + UNEMP`

The syntax `-1` removes the intercept from the model. By default, an intercept is included in the model, but - 1 explicitly excludes it.

---

## The LASSO and Longley Data

Here are some of the important observations from using LASSO regression on the Longley dataset:

- We removed the near linearly dependent features from our model.
- We improved the condition number of the data by 4 orders of magnitude.
- There is only one variable whose condfidence interval contains 0.

## Flexible Modeling

To look at model selection in practice, we will consider another famous dataset.

The Guerry dataset is a collection of historical data used in support of Andre-Michel Guerry’s 1833 "Essay on the Moral Statistics of France."

>Andre-Michel Guerry’s (1833) Essai sur la Statistique Morale
de la France was one of the foundation studies of modern social science.
Guerry assembled data on crimes, suicides, literacy and other “moral
statistics,” and used tables and maps to analyze a variety of social issues
in perhaps the first comprehensive study relating such variables.

---

>Guerry’s results were startling for two reasons.
First he showed that rates of crime and suicide remained
remarkably stable over time, when broken
down by age, sex, region of France and even season
of the year; yet these numbers varied systematically
across departements of France. This regularity
of social numbers created the possibility to
conceive, for the first time, that human actions in
the social world were governed by social laws, just
as inanimate objects were governed by laws of the
physical world.

Source: "A.-M. Guerry’s Moral Statistics of France: Challenges for Multivariable
Spatial Analysis", Michael Friendly.  Statistical Science 2007, Vol. 22, No. 3, 368–399.

--- 

Here is the dataset.

In [ ]:
# Lottery is per-capital wager on Royal Lottery
df = sm.datasets.get_rdataset("Guerry", "HistData").data
df = df[['Lottery', 'Literacy', 'Wealth', 'Region']].dropna()
df.head()

---

Here is a regression using the feature `Literacy`, `Wealth`, and `Region`.

In [ ]:
mod = smf.ols(formula='Lottery ~ Literacy + Wealth + Region', data=df)
res = mod.fit()
print(res.summary())

---

In the previous cell, using the patsy syntax determined that elements of `Region` were text strings, so it treated `Region` as a categorical variable. 

Alternatively, we could manually enforce this with the syntax on the following slide. Recall that the `-` sign is used to remove columns/variables. Here we remove the intercept from a model by.

---

In [ ]:
res = smf.ols(formula='Lottery ~ Literacy + Wealth + C(Region) -1 ', data=df).fit()
print(res.summary())

---

We can also apply vectorized functions to the variables in our model. The following cell shows how to do this. In this case we apply the natural log function to the `Literacy` column and use this single column to predict the `Lottery` values.

In [ ]:
res = smf.ols(formula='Lottery ~ np.log(Literacy)', data=df).fit()
print(res.summary())

## Recap

We discussed how to perform regularization in linear regression to avoid issues of overfitting due to multicollinearity.

We discussed how the condition number of a matrix indicates whether we have issues with multicollinearity. 

We saw that large condition numbers indicate multicollinearity.

To address this issue we considered both Ridge and LASSO regression.

We also learned about the patsy syntax in the statsmodel package.